# Employee Salary Prediction

Predict an employee's salary based on experience, education, role, and company attributes.

**Suggested real dataset (Kaggle):** search for `Salary Prediction Dataset / Data Science Job Salaries`. Download it and place the CSV as `data/employee_salary.csv` next to this notebook to use real data automatically.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
np.random.seed(42)


## 2. Load Data

In [ ]:
import os

DATA_PATH = "data/employee_salary.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded real dataset from {DATA_PATH}, shape={df.shape}")
else:
    print("No local dataset found at 'data/employee_salary.csv'.")
    print("To use real data: search Kaggle for 'Salary Prediction Dataset / Data Science Job Salaries',")
    print("download it, place the CSV at 'employee_salary.csv' inside a 'data/' folder next to this notebook,")
    print("and re-run this cell.")
    print("Generating a realistic synthetic dataset instead so the notebook runs end-to-end...\n")

    n = 1800
    experience = np.random.uniform(0, 30, n)
    education = np.random.choice(['bachelor','master','phd'], n, p=[0.55,0.35,0.1])
    role = np.random.choice(['engineer','manager','analyst','designer','sales'], n)
    company_size = np.random.choice(['startup','mid','large'], n, p=[0.3,0.4,0.3])
    location = np.random.choice(['urban','suburban','remote'], n, p=[0.5,0.3,0.2])
    df = pd.DataFrame({
        'years_experience': experience,
        'education_level': education,
        'job_role': role,
        'company_size': company_size,
        'location_type': location,
    })
    edu_bonus = df['education_level'].map({'bachelor':0,'master':8000,'phd':18000})
    role_base = df['job_role'].map({'engineer':70000,'manager':85000,'analyst':60000,'designer':58000,'sales':55000})
    size_mult = df['company_size'].map({'startup':0.9,'mid':1.0,'large':1.15})
    df['salary'] = (
        (role_base + df['years_experience'] * 1800 + edu_bonus) * size_mult
        + np.random.normal(0, 5000, n)
    ).clip(lower=25000)

    print(f"Synthetic dataset generated, shape={df.shape}")

df.head()


## 3. Exploratory Data Analysis

In [ ]:
print(df.info())
print()
print(df.describe(include='all').T)


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['salary'], kde=True)
plt.title('Target distribution: salary')
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap (numeric features)')
plt.show()


## 4. Preprocessing

In [ ]:
df = df.dropna().reset_index(drop=True)

categorical_cols = ['education_level', 'job_role', 'company_size', 'location_type']
le_dict = {}
for c in categorical_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str))
    le_dict[c] = le

X = df.drop(columns=['salary'])
y = df['salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 5. Model Training & Comparison

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42)
}

results = {}
for name, model in models.items():
    if name == "Linear Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    print(f"{name:20s} | MAE: {mae:,.2f} | RMSE: {rmse:,.2f} | R2: {r2:.3f}")

best_name = max(results, key=lambda k: results[k]["R2"])
best_model = models[best_name]
print(f"\nBest model: {best_name}")


## 6. Actual vs Predicted

In [ ]:
final_preds = best_model.predict(X_test) if best_name == "Random Forest" else best_model.predict(X_test_scaled)

plt.figure(figsize=(6,6))
plt.scatter(y_test, final_preds, alpha=0.5)
lims = [min(y_test.min(), final_preds.min()), max(y_test.max(), final_preds.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title(f'Actual vs Predicted - {best_name}')
plt.show()


## 7. Feature Importance

In [ ]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.show()

importances


## 8. Conclusion

- Compared a simple linear/logistic baseline against a Random Forest.
- The Random Forest typically captures non-linear relationships better on this kind of tabular data.
- Next steps: hyperparameter tuning (GridSearchCV/RandomizedSearchCV), cross-validation, trying gradient boosting (XGBoost/LightGBM), and more feature engineering once real Kaggle data is loaded.